# M2.S4 — Introduction to GPU and Accelerator Computing
## Interactive HPC notebook

This notebook accompanies **M2.S4 — Introduction to GPU and Accelerator Computing**.

The session focuses on four ideas:

1. **GPU-friendly work** — lots of similar independent operations;
2. **data movement** — a fast kernel is not enough if transfers dominate;
3. **CUDA execution** — grid → blocks → threads;
4. **programming choices** — optimized library, OpenACC, or CUDA.

### Classroom method
> **PREDICT → RUN → OBSERVE → EXPLAIN**

### Execution model

Most reasoning activities run directly in Jupyter.

The **real GPU experiment uses Slurm** because the GPU is a shared cluster resource and must be explicitly allocated.

> **small concept demo → direct Jupyter**  
> **real GPU execution → Slurm `gpu` partition**

The GPU section is designed to be robust:
- it always checks the allocated GPU with `nvidia-smi`;
- it runs CUDA if `nvcc` is available;
- it runs OpenACC if NVIDIA HPC SDK `nvc` is available;
- it clearly reports when the compiler toolchain is not exposed in the current environment.

> **Notebook build: M2S4-2026-09-20-v1**
>
> If you replace an older local copy, use **Kernel → Restart Kernel and Run All Cells**.

# 0 — Environment check

### Predict
Before running:

- Will this Jupyter kernel itself have a GPU?
- Is seeing `nvidia-smi` enough to prove that the notebook has a GPU allocation?
- Which tool allocates the GPU on the SciTech cluster?

In [ ]:
import os
import platform
import shutil
import subprocess
import time
from pathlib import Path

print("Notebook build: M2S4-2026-09-20-v1")
print("Host:", platform.node())
print("User:", os.environ.get("USER", "unknown"))
print("nvidia-smi visible:", shutil.which("nvidia-smi"))
print("nvcc visible:", shutil.which("nvcc"))
print("nvc visible:", shutil.which("nvc"))
print("sbatch visible:", shutil.which("sbatch"))
print("sinfo visible:", shutil.which("sinfo"))

In [ ]:
def submit_slurm(script_path):
    if shutil.which("sbatch") is None:
        print("sbatch is not available here. Run the GPU section in the SciTech HPC environment.")
        return None
    p = subprocess.run(["sbatch", "--parsable", script_path],
                       capture_output=True, text=True)
    if p.returncode != 0:
        print("Slurm submission failed:")
        print(p.stderr)
        return None
    job_id = p.stdout.strip().split(";")[0]
    print("Submitted Slurm job:", job_id)
    return job_id

def slurm_status(job_id):
    if not job_id:
        print("No job id is available.")
        return
    if shutil.which("squeue"):
        p = subprocess.run(
            ["squeue", "-j", str(job_id), "-o", "%.18i %.9T %.10M %.6D %R"],
            capture_output=True, text=True
        )
        print(p.stdout if p.stdout.strip()
              else f"Job {job_id} is no longer in squeue (it may have completed).")
    else:
        print("squeue is not available.")

def show_job_output(job_id, prefix):
    if not job_id:
        print("No job id is available.")
        return
    path = f"{prefix}-{job_id}.out"
    if os.path.exists(path):
        print(open(path).read())
    else:
        print(f"{path} does not exist yet. Re-run this cell after the job finishes.")

### Explain

Why can the Jupyter host show NVIDIA software while still not giving your notebook permission to use a GPU?

<details>
<summary><strong>Show explanation</strong></summary>

**Suggested explanation:** software such as `nvidia-smi` or CUDA libraries can exist on a system without your process owning a GPU. On a shared HPC cluster, Slurm controls access to the accelerator. A real GPU experiment should therefore request a GPU resource explicitly.

</details>

# 1 — Which workloads belong on a GPU?
### Slide connection: CPU latency vs GPU throughput

Classify these before revealing the answer:

| Workload | CPU or GPU first? |
|---|---|
| A. 100 million independent vector additions | ? |
| B. Tree traversal with unpredictable branches | ? |
| C. One tiny 4×4 matrix multiplication | ? |
| D. Apply the same filter to 20 million pixels | ? |

### Predict
For each case, identify:

- amount of parallel work;
- regularity of control flow;
- problem size;
- likely data-movement cost.

In [ ]:
workloads = {
    "A": {"parallelism": "very high", "regular": True,  "size": "very large"},
    "B": {"parallelism": "irregular", "regular": False, "size": "depends"},
    "C": {"parallelism": "small",      "regular": True,  "size": "tiny"},
    "D": {"parallelism": "very high", "regular": True,  "size": "very large"},
}

for name, info in workloads.items():
    print(name, "->", info)

### Explain

<details>
<summary><strong>Show explanation</strong></summary>

- **A → GPU-friendly.** Huge numbers of independent, regular operations are a good throughput workload.
- **B → usually CPU-first.** Unpredictable branching and irregular traversal are harder to keep thousands of GPU threads busy efficiently.
- **C → CPU-first.** The computation is too small for GPU launch and transfer overhead to make sense.
- **D → GPU-friendly.** The same operation over millions of pixels exposes massive data parallelism.

A GPU is not automatically faster. It is faster for the **right structure and size of work**.

</details>

# 2 — CUDA grid → blocks → threads
### Slide connection: one kernel, many threads

Suppose:

- `N = 1000` elements
- `256` threads per block

### Predict
1. How many blocks are needed?
2. How many CUDA threads are launched in total?
3. Why is `if (i < n)` still required?

In [ ]:
def launch_geometry(n, threads_per_block):
    blocks = (n + threads_per_block - 1) // threads_per_block
    launched_threads = blocks * threads_per_block
    extra_threads = launched_threads - n
    return blocks, launched_threads, extra_threads

for n in [1000, 1024, 1025, 1_000_000]:
    blocks, launched, extra = launch_geometry(n, 256)
    print(f"N={n:>9,} -> blocks={blocks:>5}, launched threads={launched:>9,}, extra={extra}")

### Observe

For `N=1000`, four blocks launch **1024 threads**.

### Explain

<details>
<summary><strong>Show explanation</strong></summary>

The last block is only partially useful. Some threads compute an index larger than 999. The boundary test:

```c
if (i < n)
```

prevents those extra threads from accessing memory outside the vector.

</details>

# 3 — The hidden tax: data movement
### Slide connection: kernel time ≠ application time

Use the simple model:

[
T_{GPU}=T_{H\to D}+T_{kernel}+T_{D\to H}
]

### Predict

Case A:
- copy input: 12 ms
- kernel: 2 ms
- copy result: 12 ms
- CPU-only: 20 ms

Case B:
- copy input: 12 ms
- kernel: 60 ms
- copy result: 12 ms
- CPU-only: 200 ms

Which implementation wins in each case?

In [ ]:
cases = {
    "A": {"h2d": 12, "kernel": 2,  "d2h": 12, "cpu": 20},
    "B": {"h2d": 12, "kernel": 60, "d2h": 12, "cpu": 200},
}

for name, c in cases.items():
    gpu_total = c["h2d"] + c["kernel"] + c["d2h"]
    winner = "GPU" if gpu_total < c["cpu"] else "CPU"
    print(f"Case {name}: GPU total={gpu_total} ms, CPU={c['cpu']} ms -> {winner} wins")

### Explain

Why can a kernel be 10× faster while the whole application improves only slightly?

<details>
<summary><strong>Show explanation</strong></summary>

Because users experience **end-to-end application time**, not kernel time in isolation. Host→device copies, device→host copies, allocation, synchronization and launch overhead remain part of the application.

The most useful GPU design often follows:

> **move once → compute many times → return once**

</details>

# 4 — Prepare a CUDA vector-add program
### Slide connection: explicit CUDA workflow

The code below writes a small CUDA program to the notebook directory.

It implements:

```text
CPU host arrays
   ↓ copy
GPU arrays
   ↓
CUDA kernel
   ↓ copy
CPU result
```

### Predict
For 1024 elements and 256 threads/block, how many blocks will the program launch?

In [ ]:
cuda_vector_add = r'''
#include <cuda_runtime.h>
#include <cstdio>
#include <cstdlib>

#define CUDA_CHECK(call) do {                                      \
    cudaError_t e = (call);                                        \
    if (e != cudaSuccess) {                                        \
        std::fprintf(stderr, "CUDA error: %s\n",                  \
                     cudaGetErrorString(e));                        \
        return 2;                                                   \
    }                                                              \
} while (0)

__global__ void add(const float *a, const float *b, float *c, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) c[i] = a[i] + b[i];
}

int main() {
    const int n = 1024;
    const int threads = 256;
    const int blocks = (n + threads - 1) / threads;
    const size_t bytes = (size_t)n * sizeof(float);

    float *a = (float*)malloc(bytes);
    float *b = (float*)malloc(bytes);
    float *c = (float*)malloc(bytes);

    for (int i = 0; i < n; ++i) {
        a[i] = (float)i;
        b[i] = (float)(2*i);
    }

    float *da, *db, *dc;
    CUDA_CHECK(cudaMalloc(&da, bytes));
    CUDA_CHECK(cudaMalloc(&db, bytes));
    CUDA_CHECK(cudaMalloc(&dc, bytes));

    CUDA_CHECK(cudaMemcpy(da, a, bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(db, b, bytes, cudaMemcpyHostToDevice));

    add<<<blocks, threads>>>(da, db, dc, n);
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaDeviceSynchronize());

    CUDA_CHECK(cudaMemcpy(c, dc, bytes, cudaMemcpyDeviceToHost));

    bool ok = true;
    for (int i = 0; i < n; ++i)
        if (c[i] != a[i] + b[i]) { ok = false; break; }

    std::printf("CUDA vector addition\n");
    std::printf("N=%d, threads/block=%d, blocks=%d\n", n, threads, blocks);
    std::printf("Result check: %s\n", ok ? "PASS" : "FAIL");
    std::printf("c[5] = %.0f\n", c[5]);

    cudaFree(da); cudaFree(db); cudaFree(dc);
    free(a); free(b); free(c);
    return ok ? 0 : 1;
}
'''

Path("m2s4_vector_add.cu").write_text(cuda_vector_add)
print("Wrote m2s4_vector_add.cu")

# 5 — Prepare a CPU vs GPU benchmark
### Slide connection: measure end-to-end

The next program compares:

- CPU computation;
- GPU kernel-only time;
- GPU end-to-end time including copies.

It tests small, medium and large vectors.

The exact crossover depends on the hardware. The important result is the **pattern**.

In [ ]:
cuda_benchmark = r'''
#include <cuda_runtime.h>
#include <algorithm>
#include <chrono>
#include <cmath>
#include <cstdio>
#include <cstdlib>
#include <vector>

#define CUDA_CHECK(call) do {                                      \
    cudaError_t e = (call);                                        \
    if (e != cudaSuccess) {                                        \
        std::fprintf(stderr, "CUDA error: %s\n",                  \
                     cudaGetErrorString(e));                        \
        std::exit(2);                                               \
    }                                                              \
} while (0)

__global__ void add(const float *a, const float *b, float *c, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) c[i] = a[i] + b[i];
}

static double cpu_add(const std::vector<float>& a,
                      const std::vector<float>& b,
                      std::vector<float>& c,
                      int reps) {
    auto t0 = std::chrono::steady_clock::now();
    for (int r = 0; r < reps; ++r)
        for (size_t i = 0; i < a.size(); ++i)
            c[i] = a[i] + b[i];
    auto t1 = std::chrono::steady_clock::now();
    return std::chrono::duration<double, std::milli>(t1-t0).count();
}

static void run_case(int n, int reps) {
    size_t bytes = (size_t)n * sizeof(float);
    std::vector<float> a(n,1.0f), b(n,2.0f), cpu(n), gpu(n);

    double cpu_ms = cpu_add(a,b,cpu,reps);

    float *da,*db,*dc;
    CUDA_CHECK(cudaMalloc(&da,bytes));
    CUDA_CHECK(cudaMalloc(&db,bytes));
    CUDA_CHECK(cudaMalloc(&dc,bytes));

    int threads=256;
    int blocks=(n+threads-1)/threads;

    CUDA_CHECK(cudaMemcpy(da,a.data(),bytes,cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(db,b.data(),bytes,cudaMemcpyHostToDevice));

    cudaEvent_t s,e;
    CUDA_CHECK(cudaEventCreate(&s));
    CUDA_CHECK(cudaEventCreate(&e));
    CUDA_CHECK(cudaEventRecord(s));
    for(int r=0;r<reps;++r) add<<<blocks,threads>>>(da,db,dc,n);
    CUDA_CHECK(cudaEventRecord(e));
    CUDA_CHECK(cudaEventSynchronize(e));
    float kernel_ms=0;
    CUDA_CHECK(cudaEventElapsedTime(&kernel_ms,s,e));

    auto t0=std::chrono::steady_clock::now();
    CUDA_CHECK(cudaMemcpy(da,a.data(),bytes,cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(db,b.data(),bytes,cudaMemcpyHostToDevice));
    for(int r=0;r<reps;++r) add<<<blocks,threads>>>(da,db,dc,n);
    CUDA_CHECK(cudaMemcpy(gpu.data(),dc,bytes,cudaMemcpyDeviceToHost));
    auto t1=std::chrono::steady_clock::now();
    double total_ms=std::chrono::duration<double,std::milli>(t1-t0).count();

    bool ok=true;
    for(int i=0;i<std::min(n,1000);++i)
        if(std::fabs(gpu[i]-3.0f)>1e-5f){ok=false;break;}

    std::printf("%-10d %-6d %12.3f %14.3f %14.3f %s\n",
                n,reps,cpu_ms,kernel_ms,total_ms,ok?"PASS":"FAIL");

    cudaEventDestroy(s); cudaEventDestroy(e);
    cudaFree(da); cudaFree(db); cudaFree(dc);
}

int main(){
    std::printf("N          reps       CPU ms  GPU kernel ms   GPU total ms check\n");
    std::printf("------------------------------------------------------------------\n");
    run_case(1024,1);
    run_case(1000000,1);
    run_case(8000000,1);
    std::printf("\nReuse: large data, 100 operations while resident\n");
    run_case(8000000,100);
    return 0;
}
'''

Path("m2s4_gpu_benchmark.cu").write_text(cuda_benchmark)
print("Wrote m2s4_gpu_benchmark.cu")

# 6 — Prepare an OpenACC version
### Slide connection: one pragma can offload a loop

CUDA gives explicit control. OpenACC asks the compiler to generate accelerator code from directives.

The example below keeps the data on the accelerator across repeated operations using a data region.

In [ ]:
openacc_src = r'''
#include <stdio.h>
#include <stdlib.h>

int main(void) {
    const int n = 1000000;
    const size_t bytes = (size_t)n * sizeof(float);
    float *a = (float*)malloc(bytes);
    float *b = (float*)malloc(bytes);
    float *c = (float*)malloc(bytes);

    for (int i=0;i<n;++i) {
        a[i]=1.0f;
        b[i]=2.0f;
        c[i]=0.0f;
    }

    #pragma acc data copyin(a[0:n], b[0:n]) copy(c[0:n])
    {
        for (int r=0;r<20;++r) {
            #pragma acc parallel loop present(a[0:n],b[0:n],c[0:n])
            for (int i=0;i<n;++i)
                c[i] = a[i] + b[i] + 0.0001f*r;
        }
    }

    printf("OpenACC result c[0]=%.4f\n", c[0]);
    free(a); free(b); free(c);
    return 0;
}
'''

Path("m2s4_openacc.c").write_text(openacc_src)
print("Wrote m2s4_openacc.c")

# 7 — Real GPU experiment on SciTech
### Request → inspect → compile → run

This is the **real accelerator section**.

The Slurm job requests:

- partition: `gpu`
- one GPU
- two CPU cores
- five minutes

Inside the allocated GPU node it will:

1. show the GPU with `nvidia-smi`;
2. check whether `nvcc` is exposed;
3. if yes, compile and run CUDA vector addition and the CPU/GPU benchmark;
4. check whether NVIDIA HPC SDK `nvc` is exposed;
5. if yes, compile and run the OpenACC example;
6. report clearly if a compiler is unavailable.

### Predict

- What GPU model do you expect?
- Will the tiny 1024-element case favor CPU or GPU end-to-end?
- Why should the repeated large-vector case make better use of the GPU?

In [ ]:
gpu_job = r'''#!/bin/bash
#SBATCH --job-name=m2s4_gpu
#SBATCH --partition=gpu
#SBATCH --gpus=1
#SBATCH --cpus-per-task=2
#SBATCH --time=00:05:00
#SBATCH --output=m2s4_gpu-%j.out

set -u

echo "============================================================"
echo "M2.S4 GPU experiment"
echo "============================================================"
echo "Host: $(hostname)"
echo "Job:  $SLURM_JOB_ID"
echo

echo "=== GPU allocated by Slurm ==="
if command -v nvidia-smi >/dev/null 2>&1; then
    nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
else
    echo "ERROR: nvidia-smi is not available on this allocated node."
fi

echo
echo "=== CUDA toolchain ==="
if command -v nvcc >/dev/null 2>&1; then
    echo "nvcc: $(command -v nvcc)"
    nvcc --version | tail -n 4

    echo
    echo "--- CUDA vector addition ---"
    if nvcc -O2 m2s4_vector_add.cu -o m2s4_vector_add; then
        ./m2s4_vector_add
    fi

    echo
    echo "--- CPU vs GPU benchmark ---"
    if nvcc -O3 m2s4_gpu_benchmark.cu -o m2s4_gpu_benchmark; then
        ./m2s4_gpu_benchmark
    fi
else
    echo "nvcc is not in PATH."
    echo "The GPU allocation is valid, but the CUDA compiler environment is not exposed."
    echo "Available CUDA/NVIDIA-related modules, if any:"
    module avail 2>&1 | grep -Ei 'cuda|nvhpc|nvidia' | head -n 30 || true
fi

echo
echo "=== OpenACC toolchain ==="
if command -v nvc >/dev/null 2>&1; then
    echo "nvc: $(command -v nvc)"
    nvc --version | head -n 3 || true
    if nvc -O2 -acc -Minfo=accel m2s4_openacc.c -o m2s4_openacc; then
        ./m2s4_openacc
    fi
else
    echo "nvc is not in PATH; skipping OpenACC execution."
fi

echo
echo "=== Optional Python GPU library check ==="
python3 - <<'PY'
try:
    import cupy as cp
    print("CuPy available:", cp.__version__)
    print("CUDA devices:", cp.cuda.runtime.getDeviceCount())
    x = cp.arange(1_000_000, dtype=cp.float32)
    y = 2*x + 1
    print("CuPy GPU check:", float(y[5].get()))
except Exception as e:
    print("CuPy not available/usable:", type(e).__name__, str(e)[:200])
PY

echo
echo "GPU experiment finished."
'''

Path("m2s4_gpu.slurm").write_text(gpu_job)
print(gpu_job)

In [ ]:
if shutil.which("sinfo") is None:
    print("Slurm is not visible here. Run this section on the SciTech HPC environment.")
else:
    print("--- GPU partition ---")
    subprocess.run(["sinfo", "-p", "gpu",
                    "-o", "%P %a %l %D %c %G"], check=False)

### Submit the GPU job

Run the next cell **once**.

The job uses `sbatch`, so Jupyter remains responsive while Slurm waits for a GPU.

In [ ]:
M2S4_JOB_ID = submit_slurm("m2s4_gpu.slurm")

### Check the queue

Re-run while the job is pending or running.

In [ ]:
slurm_status(globals().get("M2S4_JOB_ID"))

### Read the GPU result

Run this after the job completes.

In [ ]:
show_job_output(globals().get("M2S4_JOB_ID"), "m2s4_gpu")

### Observe

Look for:

- GPU model and memory;
- whether `nvcc` was available;
- CUDA vector-add **PASS**;
- CPU vs GPU timing pattern;
- whether `nvc`/OpenACC was available;
- whether CuPy was available.

### Explain

Why is this one activity appropriate for Slurm while the earlier grid/block calculations were not?

<details>
<summary><strong>Show explanation</strong></summary>

The early activities are reasoning exercises and need no accelerator resource. This section actually executes accelerator code, so it must request a GPU from the shared cluster scheduler. Slurm ensures the job receives a real GPU and prevents students from using accelerator resources they were not allocated.

</details>

# 8 — Library, OpenACC or CUDA?
### Choose the highest-level solution that solves the problem well

Match each scenario:

**A**
- 10,000-line scientific C application
- one loop consumes 70% of runtime

**B**
- new NVIDIA-specific algorithm
- needs fine control over GPU execution

**C**
- dense matrix multiplication
- standard operation already implemented efficiently

### Predict
Choose: **optimized library**, **OpenACC**, or **CUDA**.

<details>
<summary><strong>Show suggested answer</strong></summary>

- **A → OpenACC is a plausible first approach.** Small source changes and directive-based offload make it suitable for incrementally accelerating an existing scientific application.
- **B → CUDA.** NVIDIA-specific work that needs explicit thread, block, memory and execution control fits CUDA.
- **C → optimized GPU library.** If a high-quality implementation already exists, use the highest-level solution rather than rewriting matrix multiplication yourself.

</details>

# Challenge — Diagnose the acceleration

A team reports:

- CPU application: **200 ms**
- GPU kernel: **20 ms**
- complete GPU application: **150 ms**

In pairs, answer:

1. Is the kernel faster than the CPU computation?
2. Is the application 10× faster?
3. Where would you investigate first?
4. What change could help if the same data is used by 50 GPU kernels?
5. Would you choose library, OpenACC or CUDA first if the hot operation is a standard dense matrix multiplication?

<details>
<summary><strong>Show suggested solution</strong></summary>

1. Yes: 20 ms vs 200 ms is a 10× kernel-level difference.
2. No: end-to-end speedup is only `200 / 150 ≈ 1.33×`.
3. First investigate CPU↔GPU transfers, allocation, synchronization and launch overhead.
4. Keep the useful data resident on the GPU and perform many kernels before copying results back.
5. Start with an optimized GPU library when an appropriate routine already exists.

</details>

# What did we learn?

1. **GPUs optimize throughput, not single-thread latency.**
2. **GPU-friendly work is large, regular and highly parallel.**
3. **CUDA maps one kernel launch to a grid of blocks and threads.**
4. **Data movement can dominate total application time.**
5. **Measure end-to-end performance, not only kernel time.**
6. **Use Slurm when you actually need a shared GPU resource.**
7. **Prefer the highest-level suitable approach: library → OpenACC → CUDA as more control is required.**

## Bridge to the next session

The GPU is fast only if the whole system feeds it efficiently. The next sessions build on the same performance question:

> **Where is the real bottleneck?**